In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from functools import reduce

### Cargamos el archivo Excel con los datos de fondos de inversión

In [ ]:
df = pd.read_excel('../data/processed/Excel_final_fondos.xlsx', index_col= 0)

Clasificamos por niveles de riesgo según la volatilidad

In [ ]:
def clasificar_riesgo(volatilidad):
    Q1 = df['Volatilidad'].quantile(0.33)
    Q2 = df['Volatilidad'].quantile(0.67)
    if volatilidad <= Q1:
        return 'Bajo'
    elif Q1 < volatilidad <= Q2:
        return 'Medio'
    else:
        return 'Alto'

In [ ]:
# Aplicar la función de clasificación de riesgo a la columna 'Volatilidad'
df['Riesgo'] = df['Volatilidad'].map(clasificar_riesgo)

### Función `rentab_acumulada`

Calcula la rentabilidad total acumulada de un fondo desde un año específico hasta 2024, considerando comisiones anuales y el interés compuesto.

#### Parámetros:
- `row`: Serie de un fondo con rentabilidades anuales y gastos corrientes.
- `años`: Año de inicio para el cálculo de la rentabilidad (por defecto, desde 2017).
- `neta`: Si es 1, calcula la rentabilidad neta (ajustada por los gastos corrientes).
- `bruta`: Si es 1, calcula la rentabilidad bruta (sin ajuste por los gastos corrientes).

#### Retorno:
Devuelve la rentabilidad total acumulada (neta o bruta) expresada como un valor decimal.

In [ ]:
def rentab_acumulada(row: pd.Series, años: int, neta: int = 0, bruta: int = 0) -> float:
    '''
    Calculo la rentabilidad total acumulada de un fondo desde 2017 (default) o otro año superior hasta 2024. 
    Tengo en cuenta las comisiones anuales y el incremento proporcionado por el interés compuesto.
    '''
    productorio = []
    if neta == 1:
        for i in range(2025 - años, 2025):
            # Calculo la rentabilidad neta
            rentab_neta  = (1 + row[f'rent {i}']) * (1 - row['Gastos Corrientes'])
            # Lo añado a la lista
            productorio.append(rentab_neta)
    if bruta == 1:
        for i in range(2025 - años, 2025):
            productorio.append(1 + row[f'rent {i}'])
    
    # Multiplico los factores de crecimiento y le resto uno para quedarme con el porcentaje de crecimiento.
    producto_total = reduce(lambda x,y: x*y, productorio) - 1 
    return producto_total

Añadir columnas al DataFrame con las rentabilidades neta y bruta para cada horizonte temporal

In [ ]:
df['Rentabilidad neta 2 años'] = df.apply(rentab_acumulada, años=2, neta = 1, axis=1)
df['Rentabilidad bruta 2 años'] = df.apply(rentab_acumulada, años=2, bruta = 1, axis=1)

df['Rentabilidad neta 4 años'] = df.apply(rentab_acumulada, años=4, neta = 1, axis=1)
df['Rentabilidad bruta 4 años'] = df.apply(rentab_acumulada, años=4, bruta = 1, axis=1)

df['Rentabilidad neta 8 años'] = df.apply(rentab_acumulada, años=8, neta = 1, axis=1)
df['Rentabilidad bruta 8 años'] = df.apply(rentab_acumulada, años=8, bruta = 1, axis=1)


#### Función `seleccionar_fondos`

Filtra los fondos por nivel de riesgo y los ordena por rentabilidad neta en el horizonte temporal especificado, seleccionando los mejores fondos.

#### Parámetros:
- `df`: DataFrame con los datos de los fondos.
- `horizonte`: Horizonte temporal en años.
- `riesgo`: Nivel de riesgo (bajo, medio, alto).
- `n_fondos`: Número de fondos a seleccionar (por defecto 20).

#### Retorno:
DataFrame con los `n_fondos` seleccionados, ordenados por rentabilidad neta.

In [ ]:
def seleccionar_fondos(df, horizonte, riesgo, n_fondos=20):
    # Filtrar los fondos que coincidan con el nivel de riesgo especificado
    df_filtrado = df[df['Riesgo'] == riesgo].copy()
    # Ordenar por rentabilidad promedio y seleccionar los top N fondos
    df_filtrado = df_filtrado.sort_values(by=f'Rentabilidad neta {horizonte} años', ascending=False).head(n_fondos)
    return df_filtrado

### Función `calcular_rentabilidades_esperadas`

Calcula las rentabilidades mínimas, máximas y promedio de una cartera de fondos en un periodo determinado, considerando los gastos corrientes.

#### Parámetros:
- `cartera`: DataFrame con los datos de los fondos, incluyendo rentabilidades y gastos corrientes.
- `años`: Horizonte temporal en años para el cálculo.

#### Retorno:
Devuelve las rentabilidades mínima, máxima y promedio (brutas y netas, ajustadas por los gastos corrientes).


In [ ]:
def calcular_rentabilidades_esperadas(cartera, años):
    rentabilidades = cartera[[f'rent {2024 - año}' for año in range(años)]]
    
    # Calcular rentabilidad mínima y máxima usando percentiles (10% y 90%)
    rent_min = rentabilidades.quantile(0.1).mean()
    rent_max = rentabilidades.quantile(0.9).mean()

    # Calcular la rentabilidad media de todos los fondos del correspondiente nivel de riesgo en el periodo indicado
    rent_promedio = rentabilidades.mean().mean()

    # Promedio de gastos corrientes
    gasto_corriente = cartera['Gastos Corrientes'].mean()
    
    # Rentabilidades netas
    rent_min_neta = rent_min - gasto_corriente
    rent_max_neta = rent_max - gasto_corriente
    rent_promedio_neta = rent_promedio - gasto_corriente
    
    return rent_min_neta, rent_promedio_neta, rent_max_neta

Tipos de cartera:
- Bajo riesgo: Horizonte de 2 años, fondos con menor volatilidad.
- Riesgo medio: Horizonte de 4 años, fondos con volatilidad moderada.
- Alto riesgo: Horizonte de 8 años, fondos con mayor volatilidad.


In [ ]:
# Carteras para cada perfil de inversor
cartera_bajo_riesgo = seleccionar_fondos(df, horizonte=2, riesgo='Bajo')
cartera_medio_riesgo = seleccionar_fondos(df, horizonte=4, riesgo='Medio')
cartera_alto_riesgo = seleccionar_fondos(df, horizonte=8, riesgo='Alto')

# Generación de gráficas para análisis visual.
1. Rentabilidad histórica de todos los fondos (2017-2024).
2. Rentabilidad promedio de la cartera bajo riesgo (2 años).
3. Rentabilidad promedio de la cartera riesgo medio (4 años).
4. Rentabilidad promedio de la cartera alto riesgo (8 años).
5. Comparativa de rentabilidades promedio para las tres carteras.
6. Gráficas de barras para rentabilidad promedio, mínima y máxima esperadas (bruta y neta).

In [ ]:
# Definir límites comunes para las escalas de las gráficas
limites_rentabilidad_historica = (
df[[f'rent {2024 - i}' for i in range(8)]].min().min(),
df[[f'rent {2024 - i}' for i in range(8)]].max().max()
)

### 1. Gráfica: Rentabilidad histórica de todos los fondos

Esta gráfica muestra la rentabilidad histórica acumulada de todos los fondos a lo largo del tiempo.


In [ ]:
años = [2017 + i for i in range(8)]
cols = [f'rent {año}'for año in años]

In [ ]:
plt.figure(figsize=(10, 6))
for index, row in df.iterrows():
    rentabilidades = row[cols].values 
    plt.plot(años, rentabilidades, alpha = 0.4)
plt.title('Rentabilidad Histórica de Todos los Fondos (2017-2024)')
plt.xlabel('Año')
plt.ylabel('Rentabilidad')
plt.grid(True)

### 2. Gráfica: Rentabilidad promedio de la cartera bajo riesgo

Esta gráfica muestra el promedio de las rentabilidades históricas de todos los fondos que componen la cartera de un bajo nivel de riesgo.


In [ ]:

promedio_bajo = cartera_bajo_riesgo[cols].mean()
plt.figure(figsize=(10, 6))
plt.plot([2017 + i for i in range(8)], promedio_bajo, marker='o')
plt.title('Rentabilidad Historica Promedio - Cartera Bajo Riesgo (2 años)')
plt.xlabel('Año')
plt.ylabel('Rentabilidad')
plt.ylim(limites_rentabilidad_historica)
plt.grid(True)

### 3. Gráfica: Rentabilidad promedio de la cartera riesgo medio

Esta gráfica muestra el promedio de las rentabilidades históricas de todos los fondos que componen la cartera con un nivel de riesgo medio.


In [ ]:
promedio_medio = cartera_medio_riesgo[cols].mean()
plt.figure(figsize=(10, 6))
plt.plot([2017 + i for i in range(8)], promedio_medio, marker='o')
plt.title('Rentabilidad Historica Promedio - Cartera Riesgo Medio (4 años)')
plt.xlabel('Año')
plt.ylabel('Rentabilidad')
plt.ylim(limites_rentabilidad_historica)
plt.grid(True)

### 4. Gráfica: Rentabilidad promedio de la cartera alto riesgo

Esta gráfica muestra el promedio de las rentabilidades históricas de todos los fondos que componen la cartera con un alto nivel de riesgo.

In [ ]:
promedio_alto = cartera_alto_riesgo[cols].mean()
plt.figure(figsize=(10, 6))
plt.plot([2017 + i for i in range(8)], promedio_alto, marker='o')
plt.title('Rentabilidad Historica Promedio - Cartera Alto Riesgo (8 años)')
plt.xlabel('Año')
plt.ylabel('Rentabilidad')
plt.ylim(limites_rentabilidad_historica)
plt.grid(True)

### 5. Gráfica Comparativa: Rentabilidades promedio de las tres carteras

Esta gráfica compara las rentabilidades promedio históricas de las tres carteras de inversión, clasificadas por niveles de riesgo (bajo, medio, alto).


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(cols, promedio_bajo, marker='o', label='Bajo Riesgo')
plt.plot(cols, promedio_medio, marker='o', label='Medio Riesgo')
plt.plot(cols, promedio_alto, marker='o', label='Alto Riesgo')
plt.title('Comparativa Rentabilidades Promedio - Tres Carteras')
plt.xlabel('Año')
plt.ylabel('Rentabilidad')
plt.ylim(limites_rentabilidad_historica)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("../reports/figures/funds/funds_portfolios_comparison.png", dpi=300, bbox_inches="tight")

### 6. Gráfica: Rentabilidad promedio, mínima y máxima esperadas (neta)

Estas gráficas muestran las rentabilidades promedio, mínima y máxima esperada para cada cartera de inversión, lo que permite visualizar el rango de desempeño esperado para los diferentes niveles de riesgo.


In [ ]:
resultados = {
    "Bajo Riesgo": calcular_rentabilidades_esperadas(cartera_bajo_riesgo, 2),
    "Riesgo Medio": calcular_rentabilidades_esperadas(cartera_medio_riesgo, 4),
    "Alto Riesgo": calcular_rentabilidades_esperadas(cartera_alto_riesgo, 8)
}

labels = ['Rent. Min Neta', 'Rent. Media Neta', 'Rent. Max Neta']
for perfil, valores in resultados.items():
    plt.figure(figsize=(10, 6))
    plt.bar(labels, valores, color=['cyan', 'cornflowerblue', 'darkblue'])
    plt.title(f'Rentabilidad Esperada - {perfil}')
    plt.ylabel('Rentabilidad')
    plt.grid(axis='y')

    nombre_archivo = perfil.lower().replace(" ", "_")
    plt.tight_layout()
    plt.savefig(f"../reports/figures/funds/expected_return_{nombre_archivo}.png", dpi=300, bbox_inches="tight")